In [31]:
import pandas as pd
import json

df_tyrewear_flattened = pd.read_csv("untitled.csv")

# 1. Your rename mapping (Example mapping ECA/ICA bogies to a continuous 1-8 sequence)
rename_dict = {
    "eca1.bogie1": "bogie1",
    "eca1.bogie2": "bogie2",
    "ica2.bogie1": "bogie3",
    "ica2.bogie2": "bogie4",
    "ica3.bogie1": "bogie5",
    "ica3.bogie2": "bogie6",
    "eca4.bogie1": "bogie7",
    "eca4.bogie2": "bogie8",
}

# 2. Fix the NaN error when loading the JSON
df_tyrewear_flattened['parsed_json'] = df_tyrewear_flattened['Completed_responses'].apply(
    lambda x: json.loads(x) if pd.notnull(x) else {}
)

# 3. Flatten the dictionary first (using the function from earlier)
def flatten_dict(d, parent_key='', sep='.'):
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep=sep).items())
        else:
            items.append((new_key, v))
    return dict(items)

df_tyrewear_flattened['flat_dict'] = df_tyrewear_flattened['parsed_json'].apply(flatten_dict)

# 4. Rename the flattened keys
def rename_flat_keys(flat_d, mapping):
    new_d = {}
    for key, value in flat_d.items():
        new_key = key
        # Check if any of our mapping targets (e.g., 'eca1.bogie1') exist in the flat key string
        for old_str, new_str in mapping.items():
            if old_str in new_key:
                # Replace that specific part of the string
                new_key = new_key.replace(old_str, new_str)
        new_d[new_key] = value
    return new_d

# Apply the renaming function
df_tyrewear_flattened['renamed_flat_dict'] = df_tyrewear_flattened['flat_dict'].apply(
    lambda d: rename_flat_keys(d, rename_dict)
)

df_tyrewear_flattened.drop(columns=['Completed_responses', 'parsed_json', 'flat_dict'], inplace=True)
df_tyrewear_flattened.rename(columns={'renamed_flat_dict': 'Completed_responses'}, inplace=True)

df_tyrewear_flattened.head()

,Maintenanceorder_no,Functional_location,Maintenanceorder_description,Date_closed,Completed_responses
0,4000442299,RSV022,WEK3,2022-01-04 03:10:00,"{'tyre_wear.bogie1.bogie_sn': '224', 'tyre_wea..."
1,4000442380,RSV011,WEK3,NaN,{}
2,4000442442,RSV023,WEK3,2022-01-15 04:30:00,"{'tyre_wear.bogie1.bogie_sn': '254', 'tyre_wea..."
3,4000443210,RSV027,WEK1,2022-01-05 04:00:00,"{'tyre_wear.bogie1.bogie_sn': '257', 'tyre_wea..."
4,4000443563,RSV003,WEK2,NaN,{}


In [33]:
import pandas as pd
import json

df_tyrewear_flattened['parsed_json'] = df_tyrewear_flattened['Completed_responses'].apply(
    lambda x: json.loads(x) if pd.notnull(x) else {}
)

df_tyrewear_flattened['parsed_items'] = df_tyrewear_flattened['parsed_json'].apply(lambda d: list(d.items()))

### 1. FIRST DEBUG (GOOD)
# df_tyrewear_flattened.head()
# df_tyrewear_flattened = df_tyrewear_flattened.drop(columns=['Completed_responses', 'parsed_json'])
df_tyrewear_flattened.to_csv("target_format.csv", index=False)

df_exploded = df_tyrewear_flattened.explode('parsed_items').reset_index(drop=True)
df_exploded[['key', 'value']] = pd.DataFrame(df_exploded['parsed_items'].tolist(), index=df_exploded.index)

df_exploded = df_exploded.rename(columns={
    'Maintenanceorder_no': 'workorder_id',
    'Functional_location': 'RSV',
    'Maintenanceorder_description': 'work_request'
})

df_exploded.drop(columns=['parsed_items', 'parsed_json', 'Completed_responses'], inplace=True)

df_exploded = df_exploded[df_exploded['workorder_id'] == 4000442299]

key_parts = df_exploded['key'].str.split('.', expand=True)

for i in range(key_parts.shape[1], 6):
    key_parts[i] = None
    
df_exploded['eca_ica'] = key_parts[1].str.upper()
df_exploded['bogie_no'] = key_parts[2].str.extract(r'(\d+)').astype(float).astype('Int64')
df_exploded['metric_type'] = key_parts[3]

type_mapping = {
    'gearbox': 'Gearbox',
    'brake': 'Brake',
    'load_wheel': 'Load Wheel'
}

tyre_position = {
    'bottom': 'Bottom',
    'right': 'Right',
    'left': 'Left',
    "a" : "Right",
    "b" : "Left",
}

mapped_a = key_parts[3].map(type_mapping).fillna(key_parts[3]).astype(str)
mapped_b = key_parts[4].map(tyre_position).fillna(key_parts[4]).astype(str)

df_exploded['wheel_type'] = mapped_a + " " + mapped_b
df_exploded['groove_no'] = key_parts[5]

df_sn = df_exploded[df_exploded['metric_type'] == 'bogie_sn'][['workorder_id', 'eca_ica', 'bogie_no', 'value']].copy()
df_sn = df_sn.rename(columns={'value': 'bogie_sn'})

# df_sn.to_csv("target_format.csv", index=False)

# df_sn = df_sn.drop_duplicates(subset=['workorder_id', 'bogie_no'])

# df_measurements = df_exploded[
#     (df_exploded['metric_type'] != 'bogie_sn') &
#     (df_exploded['groove_no'].notnull())
# ].copy()

# df_measurements['wear_value'] = df_measurements['value'].astype(float)

# df_joined = pd.merge(
#     df_measurements,
#     df_sn,
#     on=["workorder_id", "eca_ica", "bogie_no"],
#     how="left"
# )

# df_joined['Date_closed'] = pd.to_datetime(df_joined['Date_closed'])

# df_joined['date_recorded'] = df_joined['Date_closed'].dt.strftime('%A, %d %B, %Y')
# df_joined['year'] = df_joined['Date_closed'].dt.year
# df_joined['month'] = df_joined['Date_closed'].dt.month
# df_joined['day'] = df_joined['Date_closed'].dt.day

# target_columns = [
#     "workorder_id", "RSV", "work_request", "date_recorded",
#     "eca_ica", "bogie_no", "bogie_sn", "groove_no", "wear_value",
#     "year", "month", "day", "wheel_type"
# ]

# df_target_format = df_joined[target_columns]

# # df_target_format.head(50)
# df_target_format = df_target_format[df_target_format['workorder_id'] == 4000442299]
# df_target_format.to_csv("target_format.csv", index=False)

TypeError: the JSON object must be str, bytes or bytearray, not dict